# 05 — Damage Modelling

## Main question

**How accurately can the project estimate reported aircraft-damage probability for a defined wildlife-strike scenario?**

This notebook models `INDICATED_DAMAGE` as the primary binary target. Every probability is conditional on a wildlife strike having already been reported. The model does **not** estimate the probability that a strike will occur.

### Modelling decisions carried forward from Notebook 02

- Use the locked chronological periods:
  - training: 1990–2018,
  - validation: 2019–2021,
  - final test: 2022–2024.
- Protect the final test period from fitting, tuning, threshold selection, and model selection.
- Use only the canonical feature lists approved in Notebook 02.
- Compare a geographic model using `STATE` and `FAAREGION` with an airport-aware model that additionally uses `AIRPORT_ID`.
- Treat `NUM_STRUCK` as an ordered category rather than an exact count.
- Compare a majority baseline, logistic regression, Random Forest, and XGBoost.
- Use PR-AUC as the main discrimination metric because reported damage is uncommon, while also reporting ROC-AUC, precision, recall, F1, F2, Brier score, log loss, and confusion matrices.
- Select the main operating threshold by validation F1. Also report F2 and 70%/80% recall operating points as sensitivity analyses.
- Preserve model probabilities for calibration in Notebook 06.

The notebook exports a readable pre-model dataset and a transformed model-input sample with no missing values. The transformed sample is for demonstration; learned preprocessing remains inside each fitted pipeline.


## 1. Environment and reproducibility

`xgboost` is installed explicitly because it is not part of the Python standard library or scikit-learn. The command is safe to rerun in Jupyter environments.


In [ ]:
%pip install -q xgboost

In [ ]:
from pathlib import Path
import json
import sys
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score, accuracy_score,
    f1_score, fbeta_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_recall_curve, brier_score_loss, log_loss
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)


## 2. Portable paths

The notebook searches upward from the current working directory. It also accepts the uploaded filenames used during review, which makes the notebook easier to test outside the final repository structure.


In [ ]:
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data").exists() or (candidate / "faa_strikes_binary_model.csv").exists():
        ROOT = candidate
        break

PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs" / "05_damage_modelling"
MODEL_DIR = ROOT / "models" / "candidates"
DOCS_DIR = ROOT / "docs"

for directory in [PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

binary_candidates = [
    PROCESSED_DIR / "faa_strikes_binary_model.csv",
    ROOT / "faa_strikes_binary_model.csv",
    Path("/mnt/data/faa_strikes_binary_model(1).csv"),
]

feature_candidates = [
    DOCS_DIR / "feature_lists.json",
    ROOT / "feature_lists.json",
]

BINARY_PATH = next((p for p in binary_candidates if p.exists()), None)
FEATURE_LIST_PATH = next((p for p in feature_candidates if p.exists()), None)

if BINARY_PATH is None:
    raise FileNotFoundError(
        "faa_strikes_binary_model.csv was not found. Run Notebook 02 first "
        "or place its export in data/processed/."
    )

print("Repository root:", ROOT)
print("Binary dataset:", BINARY_PATH)
print("Feature-list artifact:", FEATURE_LIST_PATH or "not found; verified fallback will be used")

## 3. Load and verify the restricted modelling dataset

The binary file is a **pre-model dataset**, not the final numerical matrix. It may retain missing numerical values because imputation must be learned from the training period inside each pipeline.

The checks below demonstrate its dimensions, memory use, temporal coverage, target prevalence, and missingness.


In [ ]:
df = pd.read_csv(BINARY_PATH, low_memory=False)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Approximate memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
display(df.head())
display(df.info())

In [ ]:
required_columns = {
    "INDICATED_DAMAGE", "TEMPORAL_SPLIT", "INCIDENT_YEAR",
    "INDEX_NR", "TARGET_CONFLICT_FLAG"
}
missing_required = sorted(required_columns - set(df.columns))
assert not missing_required, f"Missing required columns: {missing_required}"

assert set(df["INDICATED_DAMAGE"].dropna().unique()).issubset({0, 1})
assert df["INDICATED_DAMAGE"].notna().all()
assert df["INDEX_NR"].is_unique, "INDEX_NR should identify one retained incident."

split_summary = (
    df.groupby("TEMPORAL_SPLIT")
      .agg(
          rows=("INDEX_NR", "size"),
          first_year=("INCIDENT_YEAR", "min"),
          last_year=("INCIDENT_YEAR", "max"),
          damaged=("INDICATED_DAMAGE", "sum"),
          damage_rate=("INDICATED_DAMAGE", "mean"),
      )
      .reset_index()
)
split_summary["damage_rate_pct"] = 100 * split_summary["damage_rate"]
display(split_summary.round(3))

missing_summary = (
    df.isna().mean().mul(100)
      .rename("missing_pct")
      .to_frame()
      .query("missing_pct > 0")
      .sort_values("missing_pct", ascending=False)
)
display(missing_summary.round(2))

### Interpretation

A lower damage rate in later periods is evidence of temporal distribution shift. It may reflect changes in strike circumstances, aircraft fleets, reporting practice, or data completeness. The chronological design therefore provides a more demanding and relevant assessment than a random split.

Missing values in `HEIGHT`, `SPEED`, or `NUM_ENGS` do not mean that Notebook 02 failed. They remain visible so that their imputation statistics are estimated only from training records.


## 4. Canonical feature lists and leakage assertion

Notebook 02 remains the source of truth. When its JSON artifact is available, this notebook loads it directly. The fallback lists reproduce the approved definitions and are checked against the dataset.

`INCIDENT_YEAR`, `TEMPORAL_SPLIT`, `INDEX_NR`, and `TARGET_CONFLICT_FLAG` are retained for validation or audit purposes but are not predictors.


In [ ]:
fallback_feature_lists = {
    "core_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED"
    ],
    "extended_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED", "PHASE_OF_FLIGHT", "HEIGHT", "SPEED",
        "TIME_OF_DAY", "SKY", "PRECIPITATION", "STATE", "FAAREGION"
    ],
    "airport_aware_features": [
        "SEASON", "MONTH_SIN", "MONTH_COS", "WILDLIFE_TYPE", "SIZE",
        "NUM_STRUCK", "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG",
        "NUM_ENGS", "WARNED", "PHASE_OF_FLIGHT", "HEIGHT", "SPEED",
        "TIME_OF_DAY", "SKY", "PRECIPITATION", "STATE", "FAAREGION",
        "AIRPORT_ID"
    ],
    "primary_target": "INDICATED_DAMAGE",
    "seed": SEED,
}

if FEATURE_LIST_PATH is not None:
    with open(FEATURE_LIST_PATH, "r", encoding="utf-8") as f:
        feature_lists = json.load(f)
else:
    feature_lists = fallback_feature_lists.copy()
    warnings.warn(
        "feature_lists.json was not found. The approved Notebook 02 fallback "
        "is being used. Regenerate the JSON artifact before final submission."
    )

TARGET = feature_lists.get("primary_target", "INDICATED_DAMAGE")
CORE_FEATURES = [c for c in feature_lists["core_features"] if c in df.columns]
EXTENDED_FEATURES = [c for c in feature_lists["extended_features"] if c in df.columns]
AIRPORT_AWARE_FEATURES = [
    c for c in feature_lists["airport_aware_features"] if c in df.columns
]

feature_set_table = pd.DataFrame({
    "feature_set": ["core", "extended_geographic", "airport_aware"],
    "feature_count": [
        len(CORE_FEATURES), len(EXTENDED_FEATURES), len(AIRPORT_AWARE_FEATURES)
    ],
    "includes_state": [
        "STATE" in CORE_FEATURES, "STATE" in EXTENDED_FEATURES,
        "STATE" in AIRPORT_AWARE_FEATURES
    ],
    "includes_region": [
        "FAAREGION" in CORE_FEATURES, "FAAREGION" in EXTENDED_FEATURES,
        "FAAREGION" in AIRPORT_AWARE_FEATURES
    ],
    "includes_airport_id": [
        "AIRPORT_ID" in CORE_FEATURES, "AIRPORT_ID" in EXTENDED_FEATURES,
        "AIRPORT_ID" in AIRPORT_AWARE_FEATURES
    ],
})
display(feature_set_table)

assert set(EXTENDED_FEATURES).issubset(AIRPORT_AWARE_FEATURES)
assert "AIRPORT_ID" not in EXTENDED_FEATURES
assert "AIRPORT_ID" in AIRPORT_AWARE_FEATURES

In [ ]:
forbidden_predictors = {
    "DAMAGE_LEVEL", "EFFECT", "EFFECT_OTHER", "AOS",
    "COST_REPAIRS", "COST_OTHER", "COST_REPAIRS_INFL_ADJ",
    "COST_OTHER_INFL_ADJ", "NR_INJURIES", "NR_FATALITIES",
    "INCIDENT_YEAR", "TEMPORAL_SPLIT", "INDEX_NR", "TARGET_CONFLICT_FLAG"
}
forbidden_predictors |= {
    c for c in df.columns if c.startswith(("DAM_", "STR_", "ING_"))
}

for set_name, features in {
    "core": CORE_FEATURES,
    "extended": EXTENDED_FEATURES,
    "airport_aware": AIRPORT_AWARE_FEATURES,
}.items():
    leaked = sorted(set(features) & forbidden_predictors)
    assert not leaked, f"{set_name} contains forbidden predictors: {leaked}"

print("Leakage assertions passed.")

## 5. Demonstration export: readable pre-model dataset

This output is the cleaned, human-readable table that individuals can inspect. It includes the target, all approved airport-aware predictors, split metadata, and identifiers. It is not falsely presented as an already imputed matrix.


In [ ]:
PREMODEL_COLUMNS = list(dict.fromkeys(
    [TARGET] + AIRPORT_AWARE_FEATURES +
    ["TEMPORAL_SPLIT", "INCIDENT_YEAR", "INDEX_NR", "TARGET_CONFLICT_FLAG"]
))

premodel_df = df[PREMODEL_COLUMNS].copy()
premodel_path = PROCESSED_DIR / "faa_strikes_binary_model_ready.csv"
premodel_df.to_csv(premodel_path, index=False)

premodel_info = pd.DataFrame([{
    "file": premodel_path.name,
    "rows": premodel_df.shape[0],
    "columns": premodel_df.shape[1],
    "file_size_mb": premodel_path.stat().st_size / 1024**2,
    "cells_missing": int(premodel_df.isna().sum().sum()),
    "purpose": "Readable cleaned dataset before learned preprocessing"
}])
display(premodel_info.round(2))
display(premodel_df.head())

## 6. Locked chronological partitions

The project retains the chronological periods established in Notebook 02:

- training: 1990–2018,
- validation: 2019–2021,
- locked final test: 2022–2024.

Hyperparameter selection is performed only within the 1990–2018 training
period. Rather than using a single development/tuning split, chronological
cross-validation is used so that each fold trains on earlier observations and
evaluates on later observations.

The 2019–2021 validation period is not used for hyperparameter selection.
The 2022–2024 final test period remains completely untouched in this notebook.


In [ ]:
train_mask = df["TEMPORAL_SPLIT"].eq("train_1990_2018")
validation_mask = df["TEMPORAL_SPLIT"].eq("validation_2019_2021")
test_mask = df["TEMPORAL_SPLIT"].eq("test_2022_2024")


y_train = df.loc[train_mask, TARGET].astype(int)
y_validation = df.loc[validation_mask, TARGET].astype(int)
y_test_locked = df.loc[test_mask, TARGET].astype(int)

partition_table = pd.DataFrame([
    [
        "training_for_chronological_cv",
        int(train_mask.sum()),
        1990,
        2018,
        df.loc[train_mask, TARGET].mean()
    ],
    [
        "official_validation",
        int(validation_mask.sum()),
        2019,
        2021,
        df.loc[validation_mask, TARGET].mean()
    ],
    [
        "locked_final_test",
        int(test_mask.sum()),
        2022,
        2024,
        df.loc[test_mask, TARGET].mean()
    ],
], columns=[
    "partition",
    "rows",
    "first_year",
    "last_year",
    "damage_rate"
])

partition_table["damage_rate_pct"] = (
    100 * partition_table["damage_rate"]
)

display(partition_table.round(3))

assert not (train_mask & validation_mask).any()
assert not (train_mask & test_mask).any()
assert not (validation_mask & test_mask).any()

### Chronological cross-validation design


In [ ]:
train_cv_df = (
    df.loc[train_mask]
      .sort_values(["INCIDENT_YEAR", "INDEX_NR"])
      .copy()
)

TRAIN_YEARS = sorted(train_cv_df["INCIDENT_YEAR"].unique())

print(
    f"Training period available for CV: "
    f"{min(TRAIN_YEARS)}–{max(TRAIN_YEARS)}"
)
print(f"Training observations: {len(train_cv_df):,}")

In [ ]:
CV_FOLDS = [
    {
        "fold": 1,
        "train_years": range(1990, 2004),
        "validation_years": range(2004, 2007),
    },
    {
        "fold": 2,
        "train_years": range(1990, 2007),
        "validation_years": range(2007, 2010),
    },
    {
        "fold": 3,
        "train_years": range(1990, 2010),
        "validation_years": range(2010, 2013),
    },
    {
        "fold": 4,
        "train_years": range(1990, 2013),
        "validation_years": range(2013, 2016),
    },
    {
        "fold": 5,
        "train_years": range(1990, 2016),
        "validation_years": range(2016, 2019),
    },
]

cv_summary = []

for spec in CV_FOLDS:
    fold_train_mask = (
        train_mask
        & df["INCIDENT_YEAR"].isin(spec["train_years"])
    )
    fold_val_mask = (
        train_mask
        & df["INCIDENT_YEAR"].isin(spec["validation_years"])
    )

    cv_summary.append({
        "fold": spec["fold"],
        "train_start": min(spec["train_years"]),
        "train_end": max(spec["train_years"]),
        "validation_start": min(spec["validation_years"]),
        "validation_end": max(spec["validation_years"]),
        "train_rows": int(fold_train_mask.sum()),
        "validation_rows": int(fold_val_mask.sum()),
        "train_damage_rate": df.loc[
            fold_train_mask, TARGET
        ].mean(),
        "validation_damage_rate": df.loc[
            fold_val_mask, TARGET
        ].mean(),
    })

cv_summary = pd.DataFrame(cv_summary)
display(cv_summary.round(4))

Chronological cross-validation differs from ordinary random cross-validation.
Rows are ordered by time, and each validation fold occurs after the records
used to fit that fold.

This prevents future training-period observations from being used to predict
earlier observations. Preprocessing remains inside the pipeline, so
imputation, scaling, and category encoding are also fitted separately within
each training fold.


## 7. Preprocessing design

`NUM_STRUCK` is represented by its documented order:

1. `1`
2. `2–10`
3. `11–100`
4. `More than 100`

`Unknown` and `Not reported` are not assigned artificial counts. After most-frequent imputation, the ordinal encoder maps the observed ranges to `0`–`3`, preserving their order from one reported strike through more than 100. The value `-1` is reserved for an unexpected category not observed during fitting. The encoded values represent ordered groups, not exact wildlife counts.

Other categorical variables are one-hot encoded. Rare categories are grouped using a threshold learned from training data only. Numerical variables are median-imputed. Logistic regression also standardizes numerical inputs.

The one-hot output remains sparse, which is important for `AIRPORT_ID`.


In [ ]:
NUM_STRUCK_ORDER = [["1", "2–10", "11–100", "More than 100"]]

def make_preprocessor(features, scale_numeric=False, min_category_frequency=50):
    ordinal_features = [c for c in ["NUM_STRUCK"] if c in features]
    numeric_features = [
        c for c in features
        if c != "NUM_STRUCK" and pd.api.types.is_numeric_dtype(df[c])
    ]
    categorical_features = [
        c for c in features
        if c not in ordinal_features + numeric_features
    ]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler(with_mean=False)))

    transformers = []
    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline(numeric_steps),
            numeric_features
        ))
    if ordinal_features:
        transformers.append((
            "num_struck_ordinal",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ordinal", OrdinalEncoder(
                    categories=NUM_STRUCK_ORDER,
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1
                ))
            ]),
            ordinal_features
        ))
    if categorical_features:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="constant", fill_value="Not reported"
                )),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=min_category_frequency,
                    sparse_output=True
                ))
            ]),
            categorical_features
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=True
    )

preprocessing_design = pd.DataFrame([
    ["NUM_STRUCK", "Ordinal encoding", "Ordered ranges; unknown/unreported remain separate"],
    ["Other categorical fields", "Impute explicit label + sparse one-hot", "Nominal categories"],
    ["Numerical fields", "Training median", "No global imputation"],
    ["Rare categories", "Training-fitted grouping", "Reduces sparse unstable columns"],
], columns=["field_type", "treatment", "reason"])
display(preprocessing_design)

## 8. Candidate model construction

Four modelling references are retained:

- **Majority-prior baseline** establishes no-skill probability performance.
- **Logistic regression** provides an interpretable linear baseline.
- **Random Forest** captures non-linearities and interactions without requiring them to be specified manually.
- **XGBoost** provides a regularized gradient-boosting comparison and can consume the sparse encoded design matrix efficiently.

Hyperparameters are no longer defined in this section. They are specified in Section 10 and tuned separately by model family so that runtime, parameter compatibility, and model-specific decisions remain transparent.

The updated mappings and feature lists created upstream remain the source of truth. This notebook does not redefine wildlife, aircraft, geographic, or leakage mappings during model fitting.


In [ ]:
def class_ratio(y):
    """Return the negative-to-positive class ratio from training observations only."""
    negatives = int((y == 0).sum())
    positives = int((y == 1).sum())
    if positives == 0:
        raise ValueError("The training fold contains no positive damage observations.")
    return negatives / positives


def make_pipeline(model_name, features, y_for_weight):
    """
    Construct a fresh preprocessing + estimator pipeline.

    Model-specific hyperparameters are deliberately left at stable defaults here
    because Section 10 supplies the values tested during chronological CV.
    """
    if model_name == "logistic":
        model = LogisticRegression(
            max_iter=2000,
            solver="liblinear",
            penalty="l2",
            class_weight=None,
            random_state=SEED
        )
        prep = make_preprocessor(features, scale_numeric=True)

    elif model_name == "random_forest":
        model = RandomForestClassifier(
            n_estimators=300,
            class_weight=None,
            n_jobs=-1,
            random_state=SEED
        )
        prep = make_preprocessor(features, scale_numeric=False)

    elif model_name == "xgboost":
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
            scale_pos_weight=1.0
        )
        prep = make_preprocessor(features, scale_numeric=False)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline([
        ("preprocess", prep),
        ("model", model)
    ])


## 9. Evaluation helpers

### 9.1 Probability metrics

PR-AUC is the primary discrimination metric because reported damage is uncommon. ROC-AUC is retained as a secondary ranking measure. Brier score and log loss are also reported because the project ultimately requires probability estimates for calibration and simulation.

### 9.2 Classification metrics

Precision, recall, F1, F2, and confusion-matrix counts are reported so that conventional classification performance is visible alongside probability metrics. During hyperparameter tuning, these metrics use a fixed threshold of **0.50** only as a common diagnostic reference. Hyperparameters are not selected from that threshold-specific F1 score; operating thresholds are selected later from the untouched 2019–2021 validation period.


In [ ]:
def probability_metrics(y_true, probabilities):
    return {
        "pr_auc": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
        "log_loss": log_loss(y_true, probabilities, labels=[0, 1]),
    }


def classification_metrics_at_threshold(y_true, probabilities, threshold=0.5):
    """Return classification-report metrics and confusion-matrix counts."""
    predictions = (np.asarray(probabilities) >= threshold).astype(int)

    report = classification_report(
        y_true,
        predictions,
        output_dict=True,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
        "support_damage": int(report["1"]["support"]),
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions, zero_division=0),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "f2": fbeta_score(y_true, predictions, beta=2, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def threshold_table(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    table = pd.DataFrame({
        "threshold": thresholds,
        "precision": precision[:-1],
        "recall": recall[:-1],
    })

    denom_f1 = table["precision"] + table["recall"]
    table["f1"] = np.where(
        denom_f1 > 0,
        2 * table["precision"] * table["recall"] / denom_f1,
        0
    )

    beta2 = 4
    denom_f2 = beta2 * table["precision"] + table["recall"]
    table["f2"] = np.where(
        denom_f2 > 0,
        (1 + beta2) * table["precision"] * table["recall"] / denom_f2,
        0
    )
    return table


def choose_thresholds(y_true, probabilities):
    table = threshold_table(y_true, probabilities)
    rows = []

    f1_row = table.loc[table["f1"].idxmax()].copy()
    f1_row["rule"] = "maximum_validation_f1"
    rows.append(f1_row)

    f2_row = table.loc[table["f2"].idxmax()].copy()
    f2_row["rule"] = "maximum_validation_f2_sensitivity"
    rows.append(f2_row)

    for target_recall in [0.70, 0.80]:
        eligible = table.loc[table["recall"] >= target_recall]
        if len(eligible):
            row = eligible.sort_values(
                ["precision", "threshold"], ascending=False
            ).iloc[0].copy()
            row["rule"] = f"highest_precision_at_recall_{int(target_recall * 100)}"
            rows.append(row)

    return pd.DataFrame(rows)[
        ["rule", "threshold", "precision", "recall", "f1", "f2"]
    ].reset_index(drop=True)


## 10. Hyperparameter search spaces

The hyperparameter grids were designed to explore meaningful differences in
model complexity, regularization, class-imbalance handling, and stochastic
sampling while keeping the search interpretable.

The values are not presented as universally optimal settings. Instead, they
represent a bounded set of lower, moderate, and higher-complexity alternatives
that are plausible for this dataset. Each configuration is evaluated through
the same chronological cross-validation design, so model settings are compared
under consistent temporal conditions.

### Logistic regression

The logistic-regression grid evaluates four main modelling choices:

- `C = [0.1, 1.0, 5.0]` spans stronger, default-scale, and weaker
  regularization. Smaller values constrain coefficient magnitude more
  strongly, while larger values allow a more flexible fitted relationship.
- `penalty = ["l1", "l2"]` compares sparse coefficient regularization with
  conventional squared-coefficient regularization.
- `solver = ["liblinear", "lbfgs"]` compares two commonly used optimization
  methods. The grid is split into compatible combinations because `lbfgs`
  does not support the L1 penalty in this configuration.
- `class_weight = [None, "balanced"]` evaluates whether explicit correction
  for the rare damage class improves minority-class detection. This is
  especially relevant because the target is strongly imbalanced.

The grid therefore tests whether logistic-regression performance depends more
on regularization strength, penalty form, solver choice, or imbalance
correction.

### Random Forest

The Random Forest grid evaluates ensemble size, tree complexity,
regularization, feature sampling, and class weighting:

- `n_estimators = [250, 400]` compares a moderately sized forest with a larger
  ensemble. Increasing the number of trees can improve stability and reduce
  variance, but it also increases computational cost.
- `max_depth = [12, None]` compares a depth-limited forest with unrestricted
  tree growth. This tests whether stronger structural regularization improves
  temporal generalization.
- `min_samples_split = [2, 10]` compares permissive splitting with a more
  conservative requirement before creating additional branches.
- `min_samples_leaf = [2, 10]` compares relatively small leaves with stronger
  leaf-level regularization, which can reduce overfitting to rare combinations
  of categorical and numerical predictors.
- `max_features = ["sqrt", 0.5]` compares the conventional square-root rule
  with access to a broader fraction of predictors at each split. This is
  particularly relevant because the airport-aware representation contains
  many encoded predictors.
- `class_weight = [None, "balanced_subsample"]` compares ordinary fitting with
  class weighting recalculated within each bootstrap sample.

These values provide meaningful low-versus-higher complexity contrasts without
requiring a fully exhaustive search over every possible forest configuration.

### XGBoost

The XGBoost grid evaluates boosting length, tree complexity, learning rate,
stochastic sampling, regularization, and class imbalance:

- `n_estimators = [300, 600]` compares shorter and longer boosting sequences.
- `max_depth = [3, 6]` compares relatively shallow trees with moderately deeper
  interaction structures.
- `learning_rate = [0.03, 0.08]` compares slower, more conservative boosting
  with a faster update rate.
- `subsample = [0.8, 1.0]` compares stochastic row sampling against use of the
  complete training set at each boosting iteration.
- `colsample_bytree = [0.8, 1.0]` compares stochastic feature sampling against
  making all transformed predictors available to each tree.
- `min_child_weight = [1, 5]` compares more permissive child-node creation with
  a more conservative requirement for additional splits.
- `reg_lambda = [1.0, 5.0]` compares lower and stronger L2 regularization.
- `scale_pos_weight_mode = ["none", "balanced"]` compares ordinary fitting with
  imbalance-aware weighting. When `balanced` is selected, the actual
  `scale_pos_weight` value is calculated separately inside each chronological
  fold using only that fold's training observations.

The XGBoost grid therefore tests whether improved performance comes from
additional boosting capacity, stronger or weaker regularization, stochastic
row/feature sampling, or explicit treatment of the class imbalance.

### Search-design rationale

The selected values deliberately span interpretable contrasts rather than very
fine numerical increments. Because every setting is evaluated across two
feature sets and multiple chronological folds, even a moderate grid produces a
large number of model fits. The goal is therefore to test substantively
different modelling choices rather than many nearly identical parameter
values.

Hyperparameters are selected primarily using mean chronological-CV PR-AUC,
with Brier score used as a secondary probability-quality criterion. Precision,
recall, and F1 are retained as diagnostic classification measures rather than
as the primary tuning objective.


In [ ]:
LOGISTIC_PARAMETER_GRID = [
    {
        # liblinear supports L1 and L2 penalties.
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.1, 1.0, 5.0],
        "model__class_weight": [None, "balanced"],
    },
    {
        # lbfgs provides a second common solver and is paired with L2 here.
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": [0.1, 1.0, 5.0],
        "model__class_weight": [None, "balanced"],
    },
]

RANDOM_FOREST_PARAMETER_GRID = {
    # Compare a moderately sized forest with a larger ensemble.
    "model__n_estimators": [250, 400],
    "model__max_depth": [12, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [2, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__class_weight": [None, "balanced_subsample"],
}

XGBOOST_PARAMETER_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [3, 6],
    "model__learning_rate": [0.03, 0.08],
    # Compare moderate versus full row sampling.
    "model__subsample": [0.8, 1.0],
    # Compare moderate versus full feature sampling per tree.
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 5],
    "model__reg_lambda": [1.0, 5.0],
    # Compare no class weighting with a fold-specific negative/positive class ratio.
    # This notebook-level option is converted to model__scale_pos_weight inside each fold.
    "scale_pos_weight_mode": ["none", "balanced"],
}

parameter_grids = {
    "logistic": LOGISTIC_PARAMETER_GRID,
    "random_forest": RANDOM_FOREST_PARAMETER_GRID,
    "xgboost": XGBOOST_PARAMETER_GRID,
}

FEATURE_SETS = {
    "extended_geographic": EXTENDED_FEATURES,
    "airport_aware": AIRPORT_AWARE_FEATURES,
}
MODEL_NAMES = ["logistic", "random_forest", "xgboost"]

candidate_counts = []
for model_name, grid in parameter_grids.items():
    settings_per_feature_set = len(list(ParameterGrid(grid)))
    total_fits = settings_per_feature_set * len(FEATURE_SETS) * len(CV_FOLDS)
    candidate_counts.append({
        "model": model_name,
        "settings_per_feature_set": settings_per_feature_set,
        "feature_sets": len(FEATURE_SETS),
        "cv_folds": len(CV_FOLDS),
        "total_model_fits": total_fits,
    })

candidate_count_table = pd.DataFrame(candidate_counts)
display(candidate_count_table)
print(
    "Total chronological-CV model fits:",
    f"{candidate_count_table['total_model_fits'].sum():,}"
)


## 11. Majority-class baseline

The majority classifier has no feature dependence, so it is evaluated once rather than duplicated across feature sets. Its positive-class probabilities are still saved for a complete comparison.


In [ ]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
dummy_val_prob = dummy.predict_proba(
    np.zeros((len(y_validation), 1))
)[:, 1]

baseline_metrics = {
    "model": "majority_prior",
    "feature_set": "none",
    **probability_metrics(y_validation, dummy_val_prob),
    **threshold_metrics(y_validation, dummy_val_prob, threshold=0.5),
}
display(pd.DataFrame([baseline_metrics]).round(4))

## 12. Hyperparameter tuning with expanding chronological cross-validation

Hyperparameter tuning is restricted to the approved 1990–2018 training period. Each candidate setting is evaluated across the year-blocked expanding-window folds defined in Section 6.1. In every fold, preprocessing and model fitting use only the earlier training years, while scoring uses the later validation years.

The three model families are executed in separate cells. This makes runtime transparent and preserves completed results if a later, more expensive model needs to be rerun separately or moved to different hardware.

Candidate settings are ranked primarily by **mean chronological-CV PR-AUC**. Mean Brier score is used as a secondary probability-quality criterion. Precision, recall, F1, macro F1, weighted F1, and confusion counts at a fixed threshold of 0.50 are retained as diagnostic classification-report evidence.


### 12.1 Reusable chronological-CV evaluator


In [ ]:
def apply_xgboost_scale_mode(params, y_training):
    """Translate the notebook-level scale_pos_weight mode into an estimator parameter."""
    resolved = params.copy()
    scale_mode = resolved.pop("scale_pos_weight_mode", None)

    if scale_mode == "balanced":
        resolved["model__scale_pos_weight"] = class_ratio(y_training)
    elif scale_mode == "none":
        resolved["model__scale_pos_weight"] = 1.0

    return resolved


def evaluate_parameter_setting_cv(model_name, features, params):
    """Evaluate one parameter setting across all chronological CV folds."""
    fold_rows = []

    for fold_spec in CV_FOLDS:
        fold_train_mask = (
            train_mask
            & df["INCIDENT_YEAR"].isin(fold_spec["train_years"])
        )
        fold_val_mask = (
            train_mask
            & df["INCIDENT_YEAR"].isin(fold_spec["validation_years"])
        )

        X_fold_train = df.loc[fold_train_mask, features]
        y_fold_train = df.loc[fold_train_mask, TARGET].astype(int)
        X_fold_val = df.loc[fold_val_mask, features]
        y_fold_val = df.loc[fold_val_mask, TARGET].astype(int)

        pipeline = make_pipeline(model_name, features, y_fold_train)
        resolved_params = params.copy()
        if model_name == "xgboost":
            resolved_params = apply_xgboost_scale_mode(
                resolved_params,
                y_fold_train
            )

        pipeline.set_params(**resolved_params)
        pipeline.fit(X_fold_train, y_fold_train)
        probabilities = pipeline.predict_proba(X_fold_val)[:, 1]

        fold_rows.append({
            "fold": fold_spec["fold"],
            "train_start_year": min(fold_spec["train_years"]),
            "train_end_year": max(fold_spec["train_years"]),
            "validation_start_year": min(fold_spec["validation_years"]),
            "validation_end_year": max(fold_spec["validation_years"]),
            "train_rows": len(y_fold_train),
            "validation_rows": len(y_fold_val),
            "train_damage_rate": y_fold_train.mean(),
            "validation_damage_rate": y_fold_val.mean(),
            **probability_metrics(y_fold_val, probabilities),
            **classification_metrics_at_threshold(
                y_fold_val,
                probabilities,
                threshold=0.50
            ),
        })

    return pd.DataFrame(fold_rows)


def summarize_cv_setting(model_name, feature_set_name, params, fold_results):
    """Aggregate fold-level evidence for one candidate setting."""
    return {
        "model": model_name,
        "feature_set": feature_set_name,
        "parameters": json.dumps(params, sort_keys=True),
        "mean_pr_auc": fold_results["pr_auc"].mean(),
        "std_pr_auc": fold_results["pr_auc"].std(),
        "mean_roc_auc": fold_results["roc_auc"].mean(),
        "mean_brier_score": fold_results["brier_score"].mean(),
        "mean_log_loss": fold_results["log_loss"].mean(),
        "mean_accuracy": fold_results["accuracy"].mean(),
        "mean_precision": fold_results["precision"].mean(),
        "mean_recall": fold_results["recall"].mean(),
        "mean_f1": fold_results["f1"].mean(),
        "mean_macro_f1": fold_results["macro_f1"].mean(),
        "mean_weighted_f1": fold_results["weighted_f1"].mean(),
    }


### 12.2 Logistic regression tuning

This cell evaluates the expanded logistic-regression grid independently and saves its results immediately.


In [ ]:
logistic_tuning_rows = []

for feature_set_name, features in FEATURE_SETS.items():
    for params in ParameterGrid(LOGISTIC_PARAMETER_GRID):
        print("Logistic:", feature_set_name, params)
        fold_results = evaluate_parameter_setting_cv(
            "logistic", features, params
        )
        logistic_tuning_rows.append(
            summarize_cv_setting(
                "logistic", feature_set_name, params, fold_results
            )
        )

logistic_tuning_results = pd.DataFrame(logistic_tuning_rows).sort_values(
    ["mean_pr_auc", "mean_brier_score"],
    ascending=[False, True]
)
logistic_tuning_results.to_csv(
    OUTPUT_DIR / "logistic_chronological_cv_results.csv",
    index=False
)
display(logistic_tuning_results.head(50).round(4))


#### Logistic-regression inference placeholder

After execution, summarize which solver, penalty, regularization strength, and class-weight choice were most stable across chronological folds. Note whether class weighting improved recall at the cost of precision or probability quality.


### 12.3 Random Forest tuning

This cell evaluates the expanded Random Forest grid separately because tree ensembles are more computationally expensive than logistic regression.


In [ ]:
random_forest_tuning_rows = []

for feature_set_name, features in FEATURE_SETS.items():
    for params in ParameterGrid(RANDOM_FOREST_PARAMETER_GRID):
        print("Random Forest:", feature_set_name, params)
        fold_results = evaluate_parameter_setting_cv(
            "random_forest", features, params
        )
        random_forest_tuning_rows.append(
            summarize_cv_setting(
                "random_forest", feature_set_name, params, fold_results
            )
        )

random_forest_tuning_results = pd.DataFrame(
    random_forest_tuning_rows
).sort_values(
    ["mean_pr_auc", "mean_brier_score"],
    ascending=[False, True]
)
random_forest_tuning_results.to_csv(
    OUTPUT_DIR / "random_forest_chronological_cv_results.csv",
    index=False
)
display(random_forest_tuning_results.head(50).round(4))


#### Random Forest inference placeholder

After execution, describe whether shallower/deeper trees, larger leaves, feature subsampling, or class weighting improved chronological performance. Comment on whether the airport-aware feature set required stronger complexity control than the broader geographic set.


### 12.4 XGBoost tuning

This cell evaluates XGBoost independently. The `scale_pos_weight` comparison is calculated from each fold's training distribution only.


In [ ]:
xgboost_tuning_rows = []

for feature_set_name, features in FEATURE_SETS.items():
    for params in ParameterGrid(XGBOOST_PARAMETER_GRID):
        print("XGBoost:", feature_set_name, params)
        fold_results = evaluate_parameter_setting_cv(
            "xgboost", features, params
        )
        xgboost_tuning_rows.append(
            summarize_cv_setting(
                "xgboost", feature_set_name, params, fold_results
            )
        )

xgboost_tuning_results = pd.DataFrame(xgboost_tuning_rows).sort_values(
    ["mean_pr_auc", "mean_brier_score"],
    ascending=[False, True]
)
xgboost_tuning_results.to_csv(
    OUTPUT_DIR / "xgboost_chronological_cv_results.csv",
    index=False
)
display(xgboost_tuning_results.head(50).round(4))


#### XGBoost inference placeholder

After execution, summarize the preferred depth, learning rate, number of estimators, child-weight, regularization, and class-imbalance strategy. Note whether several configurations perform similarly or whether the result depends strongly on one setting.


### 12.5 Combine tuning evidence and select the best setting per model/feature set


In [ ]:
tuning_results = pd.concat(
    [
        logistic_tuning_results,
        random_forest_tuning_results,
        xgboost_tuning_results,
    ],
    ignore_index=True
).sort_values(
    ["mean_pr_auc", "mean_brier_score"],
    ascending=[False, True]
)

tuning_results.to_csv(
    OUTPUT_DIR / "internal_tuning_results.csv",
    index=False
)
display(tuning_results.head(100).round(4))

best_settings = {}
best_setting_rows = []

for model_name in MODEL_NAMES:
    for feature_set_name in FEATURE_SETS:
        candidate_df = tuning_results.loc[
            tuning_results["model"].eq(model_name)
            & tuning_results["feature_set"].eq(feature_set_name)
        ].copy()

        best_row = candidate_df.sort_values(
            ["mean_pr_auc", "mean_brier_score"],
            ascending=[False, True]
        ).iloc[0]

        best_settings[(model_name, feature_set_name)] = json.loads(
            best_row["parameters"]
        )
        best_setting_rows.append(best_row)

best_setting_summary = pd.DataFrame(best_setting_rows).reset_index(drop=True)
best_setting_summary.to_csv(
    OUTPUT_DIR / "selected_hyperparameters_from_cv.csv",
    index=False
)
display(best_setting_summary.round(4))


### Chronological-CV inference placeholder

After all three tuning cells finish, compare the best mean PR-AUC and its fold-to-fold standard deviation across model families and feature sets. Discuss Brier score/log loss alongside discrimination, and use the fixed-threshold precision/recall/F1 columns only as diagnostic evidence. Do not use the 2019–2021 validation period to revise hyperparameters.


### 12.6 Classification reports for the CV-selected settings

To make the conventional classification evidence explicit, the selected setting for each model/feature-set pair is re-evaluated across the same chronological folds. Fold predictions are pooled only for reporting at the common 0.50 threshold; this does not change hyperparameter selection.


In [ ]:
cv_classification_reports = []

for (model_name, feature_set_name), params in best_settings.items():
    features = FEATURE_SETS[feature_set_name]
    pooled_true = []
    pooled_pred = []

    for fold_spec in CV_FOLDS:
        fold_train_mask = (
            train_mask
            & df["INCIDENT_YEAR"].isin(fold_spec["train_years"])
        )
        fold_val_mask = (
            train_mask
            & df["INCIDENT_YEAR"].isin(fold_spec["validation_years"])
        )

        X_fold_train = df.loc[fold_train_mask, features]
        y_fold_train = df.loc[fold_train_mask, TARGET].astype(int)
        X_fold_val = df.loc[fold_val_mask, features]
        y_fold_val = df.loc[fold_val_mask, TARGET].astype(int)

        pipeline = make_pipeline(model_name, features, y_fold_train)
        resolved_params = params.copy()
        if model_name == "xgboost":
            resolved_params = apply_xgboost_scale_mode(
                resolved_params,
                y_fold_train
            )
        pipeline.set_params(**resolved_params)
        pipeline.fit(X_fold_train, y_fold_train)

        probabilities = pipeline.predict_proba(X_fold_val)[:, 1]
        predictions = (probabilities >= 0.50).astype(int)
        pooled_true.extend(y_fold_val.to_numpy())
        pooled_pred.extend(predictions)

    report = classification_report(
        pooled_true,
        pooled_pred,
        output_dict=True,
        zero_division=0
    )

    cv_classification_reports.append({
        "model": model_name,
        "feature_set": feature_set_name,
        "threshold": 0.50,
        "damage_precision": report["1"]["precision"],
        "damage_recall": report["1"]["recall"],
        "damage_f1": report["1"]["f1-score"],
        "damage_support": int(report["1"]["support"]),
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    })

cv_classification_report_df = pd.DataFrame(cv_classification_reports)
cv_classification_report_df.to_csv(
    OUTPUT_DIR / "cv_selected_settings_classification_report.csv",
    index=False
)
display(cv_classification_report_df.round(4))


#### Classification-report inference placeholder

After execution, compare the damage-class precision, recall, and F1 values. Explain whether class-imbalance handling shifts the balance toward recall, and reiterate that the 0.50 threshold is a diagnostic common reference rather than the final operating threshold.


## 13. Refit on 1990–2018 and evaluate on 2019–2021

After tuning, each candidate is refitted using the complete training period. Validation probabilities are saved for calibration and comparison. The final test set remains untouched.


In [ ]:
validation_rows = [baseline_metrics]
validation_predictions = pd.DataFrame({
    "INDEX_NR": df.loc[validation_mask, "INDEX_NR"].to_numpy(),
    "INCIDENT_YEAR": df.loc[validation_mask, "INCIDENT_YEAR"].to_numpy(),
    "observed_damage": y_validation.to_numpy(),
    "majority_prior_probability": dummy_val_prob,
})

fitted_candidates = {}

for feature_set_name, features in FEATURE_SETS.items():
    X_train = df.loc[train_mask, features]
    X_validation = df.loc[validation_mask, features]

    for model_name in MODEL_NAMES:
        print(f"Refitting {model_name} with {feature_set_name}...")
        pipeline = make_pipeline(model_name, features, y_train)

        # Resolve notebook-level XGBoost imbalance handling using the full
        # 1990–2018 training distribution only.
        selected_params = best_settings[(model_name, feature_set_name)].copy()
        resolved_params = selected_params.copy()
        if model_name == "xgboost":
            resolved_params = apply_xgboost_scale_mode(
                resolved_params,
                y_train
            )

        pipeline.set_params(**resolved_params)
        pipeline.fit(X_train, y_train)

        probabilities = pipeline.predict_proba(X_validation)[:, 1]
        candidate_key = f"{model_name}__{feature_set_name}"
        fitted_candidates[candidate_key] = pipeline
        validation_predictions[f"{candidate_key}_probability"] = probabilities

        candidate_thresholds = choose_thresholds(y_validation, probabilities)
        f1_threshold = float(
            candidate_thresholds.loc[
                candidate_thresholds["rule"].eq("maximum_validation_f1"),
                "threshold"
            ].iloc[0]
        )

        row = {
            "model": model_name,
            "feature_set": feature_set_name,
            **probability_metrics(y_validation, probabilities),
            **threshold_metrics(y_validation, probabilities, f1_threshold),
            "best_parameters": json.dumps(
                best_settings[(model_name, feature_set_name)],
                sort_keys=True
            ),
        }
        validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows)
validation_results.to_csv(
    OUTPUT_DIR / "validation_model_comparison.csv", index=False
)
validation_predictions.to_csv(
    OUTPUT_DIR / "validation_probability_predictions.csv", index=False
)

display(
    validation_results.sort_values(
        ["pr_auc", "brier_score"], ascending=[False, True]
    ).round(4)
)


### Validation inference placeholder

After execution, interpret the 2019–2021 validation results here. Compare every trained model against the majority-prior PR-AUC reference, then compare PR-AUC, ROC-AUC, Brier score, log loss, precision, recall, F1, and confusion counts. Avoid describing one model as universally "best" when discrimination and probability-quality metrics disagree.

Also comment on whether the damage prevalence differs from the 1990–2018 training period and why this makes calibration important.


## 14. Model-selection decision

The primary candidate is selected using the following hierarchy:

1. exclude the majority baseline;
2. identify candidates within 1% relative PR-AUC of the best candidate;
3. among those near-best candidates, prefer the lowest Brier score;
4. retain logistic regression as a calibration candidate even if it is not selected, because its probabilities and effects are interpretable;
5. record whether `AIRPORT_ID` improves validation performance enough to justify its additional dependence on known airports.

This rule does not automatically reward a negligible PR-AUC gain at the cost of substantially worse probability quality.


In [ ]:
candidate_results = validation_results.loc[
    validation_results["model"].ne("majority_prior")
].copy()

best_pr_auc = candidate_results["pr_auc"].max()
near_best = candidate_results.loc[
    candidate_results["pr_auc"] >= 0.99 * best_pr_auc
].copy()

selected_row = near_best.sort_values(
    ["brier_score", "log_loss", "pr_auc"],
    ascending=[True, True, False]
).iloc[0]

SELECTED_KEY = (
    f"{selected_row['model']}__{selected_row['feature_set']}"
)
selected_pipeline = fitted_candidates[SELECTED_KEY]

selection_summary = pd.DataFrame([{
    "selected_candidate": SELECTED_KEY,
    "validation_pr_auc": selected_row["pr_auc"],
    "validation_roc_auc": selected_row["roc_auc"],
    "validation_brier_score": selected_row["brier_score"],
    "validation_log_loss": selected_row["log_loss"],
    "selection_rule": (
        "Lowest Brier score among candidates within 1% of best validation PR-AUC"
    ),
    "final_test_used": False,
}])
display(selection_summary.round(4))

selection_summary.to_csv(
    OUTPUT_DIR / "model_selection_decision.csv", index=False
)

### Model-selection inference placeholder

After execution, explain which candidate was selected by the predefined rule: candidates must first fall within 1% of the best validation PR-AUC, after which lower Brier score and log loss are preferred. State clearly whether the selected candidate actually had the highest PR-AUC or was chosen as a discrimination/probability-quality compromise.


### Geographic comparison

A direct comparison is reported for each algorithm. A positive PR-AUC difference indicates that adding `AIRPORT_ID` improved temporal validation discrimination. This does not prove generalization to unseen airports; Notebook 06 must perform airport-held-out evaluation.


In [ ]:
geo_comparison = (
    candidate_results.pivot(
        index="model", columns="feature_set",
        values=["pr_auc", "brier_score", "log_loss"]
    )
)
geo_comparison.columns = [
    f"{metric}__{feature_set}" for metric, feature_set in geo_comparison.columns
]
geo_comparison = geo_comparison.reset_index()

geo_comparison["airport_pr_auc_gain"] = (
    geo_comparison["pr_auc__airport_aware"] -
    geo_comparison["pr_auc__extended_geographic"]
)
geo_comparison["airport_brier_change"] = (
    geo_comparison["brier_score__airport_aware"] -
    geo_comparison["brier_score__extended_geographic"]
)
display(geo_comparison.round(5))
geo_comparison.to_csv(
    OUTPUT_DIR / "airport_feature_comparison.csv", index=False
)

### Geographic-feature inference placeholder

After execution, compare the extended-geographic and airport-aware variants within each model family. Record whether adding `AIRPORT_ID` improves PR-AUC, Brier score, or log loss, and whether the effect is consistent across algorithms. Do not assume that improvement on temporal validation proves unseen-airport generalization; that question remains for Notebook 06.


## 15. Threshold behaviour for the selected candidate

The 0.50 threshold is not assumed to be appropriate for an imbalanced outcome. The validation set determines the following operating points:

- maximum F1: primary general-purpose threshold,
- maximum F2: recall-emphasized sensitivity threshold,
- highest precision while maintaining at least 70% recall,
- highest precision while maintaining at least 80% recall.

These thresholds are descriptive decision aids. Notebook 06 must assess calibration before simulation uses the probabilities.


In [ ]:
selected_prob_col = f"{SELECTED_KEY}_probability"
selected_val_prob = validation_predictions[selected_prob_col].to_numpy()

selected_thresholds = choose_thresholds(
    y_validation, selected_val_prob
)
selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_model_thresholds.csv", index=False
)
display(selected_thresholds.round(4))

curve_table = threshold_table(y_validation, selected_val_prob)
curve_table.to_csv(
    OUTPUT_DIR / "selected_model_threshold_curve.csv", index=False
)

### Threshold inference placeholder

After execution, interpret the maximum-F1, maximum-F2, 70%-recall, and 80%-recall operating points. Describe the precision-recall trade-off and avoid interpreting an uncalibrated classification threshold as a literal probability until Notebook 06 evaluates calibration.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curve_table["recall"], curve_table["precision"])
for row in selected_thresholds.itertuples():
    ax.scatter(row.recall, row.precision)
    ax.annotate(
        row.rule.replace("_", " "),
        (row.recall, row.precision),
        xytext=(5, 5), textcoords="offset points", fontsize=8
    )
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Validation precision–recall curve: {SELECTED_KEY}")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 16. Confusion matrices at meaningful thresholds


In [ ]:
threshold_lookup = dict(zip(
    selected_thresholds["rule"], selected_thresholds["threshold"]
))
rules_to_plot = [
    "maximum_validation_f1",
    "maximum_validation_f2_sensitivity",
    "highest_precision_at_recall_70",
    "highest_precision_at_recall_80",
]

for rule in rules_to_plot:
    if rule not in threshold_lookup:
        continue
    threshold = threshold_lookup[rule]
    predictions = (selected_val_prob >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions,
        labels=[0, 1],
        display_labels=["No reported damage", "Reported damage"],
        values_format=","
    )
    plt.title(f"{rule.replace('_', ' ').title()}\nthreshold = {threshold:.3f}")
    plt.tight_layout()
    plt.show()

### Confusion-matrix inference placeholder

After execution, report the exact true-negative, false-positive, false-negative, and true-positive counts at each displayed threshold. Explain how lowering the threshold changes missed-damage cases and false alarms. Do not recommend an operational threshold without an agreed cost for those two error types.


## 17. Save uncalibrated candidate pipelines for Notebook 06

The fitted candidate pipelines are saved in `models/candidates/`. These are
intermediate model artifacts rather than the final project model.

Each artifact contains both the fitted preprocessing pipeline and the fitted
classifier. Notebook 06 will load these candidates, compare calibration
methods, and perform chronological and airport-held-out validation before
saving one canonical calibrated pipeline in `models/final/`.


In [ ]:
candidate_manifest = []

for candidate_key, pipeline in fitted_candidates.items():
    artifact_path = MODEL_DIR / f"{candidate_key}_uncalibrated.joblib"
    joblib.dump(pipeline, artifact_path)
    candidate_manifest.append({
        "candidate": candidate_key,
        "path": str(artifact_path),
        "selected_for_primary_calibration": candidate_key == SELECTED_KEY,
        "file_size_mb": artifact_path.stat().st_size / 1024**2,
    })

manifest_df = pd.DataFrame(candidate_manifest)
manifest_df.to_csv(
    OUTPUT_DIR / "uncalibrated_candidate_manifest.csv", index=False
)
display(manifest_df.round(2))

## 18. Demonstration export: transformed model-input sample

This sample shows what the selected pipeline actually passes to its estimator after fitting preprocessing on the 1990–2018 training period.

The complete transformed matrix is not exported because sparse one-hot encoding—especially for airports—can create thousands of columns and a very large file. Instead, the notebook exports:

1. a reproducible sample of transformed training rows,
2. every transformed feature name,
3. dimensions and missing-value checks.

This is sufficient to demonstrate that the estimator receives a numerical matrix with no NaN values.


In [ ]:
selected_feature_set = selected_row["feature_set"]
selected_features = FEATURE_SETS[selected_feature_set]

sample_source = df.loc[train_mask, selected_features].sample(
    n=min(1000, int(train_mask.sum())),
    random_state=SEED
)

fitted_preprocessor = selected_pipeline.named_steps["preprocess"]
transformed_sample = fitted_preprocessor.transform(sample_source)
transformed_feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(transformed_sample, "toarray"):
    transformed_sample_array = transformed_sample.toarray()
else:
    transformed_sample_array = np.asarray(transformed_sample)

transformed_sample_df = pd.DataFrame(
    transformed_sample_array,
    columns=transformed_feature_names,
    index=sample_source.index
)
transformed_sample_df.insert(
    0, "INDEX_NR", df.loc[sample_source.index, "INDEX_NR"].to_numpy()
)

transformed_sample_path = (
    PROCESSED_DIR / "faa_strikes_training_matrix_sample.csv"
)
feature_manifest_path = (
    OUTPUT_DIR / "transformed_feature_manifest.csv"
)

transformed_sample_df.to_csv(transformed_sample_path, index=False)
pd.DataFrame({
    "feature_index": np.arange(len(transformed_feature_names)),
    "transformed_feature": transformed_feature_names
}).to_csv(feature_manifest_path, index=False)

transformed_info = pd.DataFrame([{
    "file": transformed_sample_path.name,
    "sample_rows": transformed_sample_df.shape[0],
    "columns_including_identifier": transformed_sample_df.shape[1],
    "transformed_model_features": len(transformed_feature_names),
    "file_size_mb": transformed_sample_path.stat().st_size / 1024**2,
    "nan_cells": int(transformed_sample_df.isna().sum().sum()),
    "purpose": "Demonstrate numerical input after training-fitted preprocessing"
}])
display(transformed_info.round(2))
display(transformed_sample_df.head())

assert transformed_sample_df.drop(columns="INDEX_NR").isna().sum().sum() == 0

### Transformed-data inference placeholder

After execution, record the transformed feature count, sample dimensions, and whether any NaN values remain. Explain that one-hot encoding can create many numerical columns from a smaller number of original predictors and that the serialized pipeline remains the authoritative preprocessing object.


## 19. Output inventory and dataset-size evidence


In [ ]:
output_inventory = []

for path, purpose in [
    (premodel_path, "Readable cleaned pre-model dataset"),
    (transformed_sample_path, "No-NaN transformed training sample"),
    (feature_manifest_path, "Complete transformed feature names"),
    (OUTPUT_DIR / "logistic_chronological_cv_results.csv", "Logistic chronological-CV tuning evidence"),
    (OUTPUT_DIR / "random_forest_chronological_cv_results.csv", "Random Forest chronological-CV tuning evidence"),
    (OUTPUT_DIR / "xgboost_chronological_cv_results.csv", "XGBoost chronological-CV tuning evidence"),
    (OUTPUT_DIR / "internal_tuning_results.csv", "Combined chronological-CV tuning evidence"),
    (OUTPUT_DIR / "selected_hyperparameters_from_cv.csv", "CV-selected hyperparameter settings"),
    (OUTPUT_DIR / "cv_selected_settings_classification_report.csv", "Fixed-threshold classification-report summary"),
    (OUTPUT_DIR / "validation_model_comparison.csv", "Validation metrics"),
    (OUTPUT_DIR / "validation_probability_predictions.csv", "Probabilities for calibration"),
    (OUTPUT_DIR / "selected_model_thresholds.csv", "Threshold decision evidence"),
    (OUTPUT_DIR / "uncalibrated_candidate_manifest.csv", "Candidate model artifacts"),
]:
    if path.exists():
        output_inventory.append({
            "artifact": path.name,
            "rows": (
                sum(1 for _ in open(path, encoding="utf-8", errors="ignore")) - 1
                if path.suffix == ".csv" else np.nan
            ),
            "file_size_mb": path.stat().st_size / 1024**2,
            "purpose": purpose,
        })

output_inventory_df = pd.DataFrame(output_inventory)
output_inventory_df.to_csv(
    OUTPUT_DIR / "notebook05_output_inventory.csv", index=False
)
display(output_inventory_df.round(2))


### Artifact inference placeholder

After execution, summarize the sizes and purposes of the readable pre-model dataset, transformed sample, feature manifest, tuning outputs, validation probabilities, thresholds, and candidate-model manifest. Confirm that candidate pipelines are saved under `models/candidates/` for Notebook 06.


## 20. Interpretation and hand-off to Notebook 06

### Main findings — complete after execution

Use this section only after the revised tuning and validation cells have run. Summarize:

1. how far each trained model exceeds the no-skill PR-AUC reference;
2. which hyperparameter settings were selected under expanding chronological CV and whether their PR-AUC was stable across folds;
3. whether logistic regression, Random Forest, or XGBoost provided the strongest discrimination;
4. whether probability-quality metrics agree with the discrimination ranking;
5. whether `AIRPORT_ID` adds consistent value beyond `STATE` and `FAAREGION`;
6. the selected validation threshold and its precision/recall trade-off;
7. which candidate pipelines should be carried into Notebook 06 for calibration and generalization checks.

### Final-test protection

Confirm after execution that the 2022–2024 final test observations were not used for hyperparameter tuning, model selection, or threshold selection. They must remain locked for the later generalization assessment.


## 21. Limitations

- The target represents **reported aircraft damage conditional on a reported wildlife strike**.
- The analysis is observational and supports prediction and association, not causal claims.
- Reporting practices and field completeness changed over time.
- Airport identity can improve fit by representing persistent local patterns, but it can also encourage memorization and poor behavior for unseen airports.
- `NUM_STRUCK` contains ordered ranges rather than exact counts.
- Class weighting changes model fitting and may affect raw calibration; this is one reason Notebook 06 is mandatory.
- Validation threshold choices do not create an operational policy. A real deployment would need explicit costs for missed damage and false alarms.
- The transformed CSV is a demonstration sample, not a replacement for the serialized preprocessing pipeline.


## References used for evaluation choices

- Davis, J., & Goadrich, M. (2006). *The Relationship Between Precision-Recall and ROC Curves*. ICML.
- Saito, T., & Rehmsmeier, M. (2015). *The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets*. PLOS ONE.
- Lipton, Z. C., Elkan, C., & Narayanaswamy, B. (2014). *Thresholding Classifiers to Maximize F1 Score*. arXiv:1402.1892 / ECML PKDD.
- Niculescu-Mizil, A., & Caruana, R. (2005). *Predicting Good Probabilities with Supervised Learning*. ICML.
- scikit-learn documentation: precision-recall curves, average precision, probability calibration, and model evaluation.
- XGBoost documentation: scikit-learn estimator interface and imbalanced binary classification parameters.

F1 is the primary threshold because the project has no defined cost ratio and seeks a balanced precision-recall operating point. F2 is presented only as a recall-emphasized sensitivity analysis. The 70% and 80% recall points make the trade-off visible without claiming that either level is inherently optimal.
